# ChemBreak 14 — Mini-Dataset MDP Learning Experiment

**Environment:** Google Cloud Notebook Enterprise  
**Target:** ChemDFM  
**Fixed dataset:** 24 tasks  
**Pipeline:** Baseline → Learning Epoch 1 → Epoch 2 → Epoch 3 → Freeze → Optimized Evaluation → Results

Run the cells from top to bottom. ChemDFM is loaded once and reused across the experiment.

In [ ]:
from pathlib import Path
import importlib, json, os, shutil, site, subprocess, sys

# Safe notebook rerun: close a previous runner/model before refreshing source.
if 'runner' in globals():
    try:
        runner.close()
    except Exception:
        pass
    del runner


# ── USER CONFIGURATION ──────────────────────────────────────────────────────
PROJECT_ID          = "rs-foundsecft-mghasemi"
REPO_URL            = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH              = "main"
PROJECT_SUBDIR      = "chembreak14"
EXPERIMENT_REVISION = "CB14_MDP_MINI24_V1"
LIVE                = True   # False = full mock/dry-run; True = ChemDFM + Vertex roles
LIVE_PROGRESS       = True   # Show per-task/per-turn metrics live in the running cell

content_root = Path('/content').resolve()
assert content_root.is_dir(), '/content unavailable — use Google Cloud Notebook Enterprise.'
if LIVE:
    assert PROJECT_ID.strip() and not PROJECT_ID.startswith('REPLACE_'), 'Set PROJECT_ID before live execution.'

storage_root = content_root / 'chembreak14_storage'
model_cache = storage_root / 'cache/huggingface/hub'
package_dir = storage_root / 'python_packages'
env_paths = {
    'HF_HOME': storage_root/'cache/huggingface',
    'HF_HUB_CACHE': model_cache,
    'HF_MODULES_CACHE': storage_root/'cache/huggingface/modules',
    'XDG_CACHE_HOME': storage_root/'cache/xdg',
    'TORCH_HOME': storage_root/'cache/torch',
    'TORCHINDUCTOR_CACHE_DIR': storage_root/'cache/torchinductor',
    'TRITON_CACHE_DIR': storage_root/'cache/triton',
    'CUDA_CACHE_PATH': storage_root/'cache/cuda',
    'PIP_CACHE_DIR': storage_root/'cache/pip',
    'TMPDIR': storage_root/'tmp',
}
for key, path in env_paths.items():
    path.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(path)
os.environ['TMP'] = os.environ['TEMP'] = str(env_paths['TMPDIR'])
(storage_root/'offload').mkdir(parents=True, exist_ok=True)
package_dir.mkdir(parents=True, exist_ok=True)
print('CB14 storage:', storage_root)
print('Model cache:', model_cache)


## C1 — Clone or update the GitHub repository

This is the same GitHub → cloud pattern used in the earlier ChemBreak notebooks. The notebook itself is only the controller; the actual CB14 code comes from the `chembreak14` folder in GitHub.

In [ ]:
checkout = content_root / 'chembreak14_repo'
def git(*args, cwd=None): subprocess.run(['git', *args], cwd=cwd, check=True)
if not (checkout/'.git').is_dir():
    git('clone','--branch',BRANCH,'--single-branch',REPO_URL,str(checkout))
else:
    git('fetch','origin',BRANCH,cwd=checkout); git('checkout',BRANCH,cwd=checkout); git('pull','--ff-only','origin',BRANCH,cwd=checkout)
PROJECT_DIR=(checkout/PROJECT_SUBDIR).resolve()
assert (PROJECT_DIR/'pyproject.toml').is_file(), f'CB14 package not found at {PROJECT_DIR}. Push the chembreak14 folder to GitHub first.'
os.chdir(PROJECT_DIR)
print('Project:',PROJECT_DIR)

## C2 — Verify the fixed 24-task dataset

CB14 does **not** randomly choose a new small dataset every run. The 24 tasks are already locked. The verification cell rebuilds the selection from the 500-task bank and checks that it reproduces exactly.

In [ ]:
sys.path.insert(0,str(PROJECT_DIR/'src'))
# A git pull may have changed CB14 source while this kernel is still alive.
# Remove previously imported CB14 modules so this run uses the files just pulled.
for module_name in [name for name in list(sys.modules) if name == 'chembreak14' or name.startswith('chembreak14.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()
from chembreak14.selection import verify_bundle
report=verify_bundle(PROJECT_DIR/'data/final_task_bank.csv',PROJECT_DIR/'data/CB14_mini24_manifest_v1.csv',PROJECT_DIR/'data/CB14_mini24_lock_v1.json')
print(json.dumps(report,indent=2,sort_keys=True))

## C3 — Install the CB14 dependency stack

Packages and model caches are installed under `/content/chembreak14_storage`, so CB14 does not reuse CB12 checkpoints or cache directories.

In [ ]:
# Use the same pinned compatibility strategy that worked in the mature ChemBreak cloud notebooks.
compatibility_specs=['transformers==4.40.2','tokenizers==0.19.1','huggingface-hub==0.23.5','safetensors==0.4.5','accelerate==0.30.1','sentencepiece==0.2.0','einops==0.8.1']
marker=package_dir/'cb14_compatibility.json'; expected={'specifications':compatibility_specs}
installed=json.loads(marker.read_text()) if marker.exists() else None
if installed!=expected:
    print('Installing CB14 compatibility stack (first run only)...')
    subprocess.run([sys.executable,'-m','pip','install','--target',str(package_dir),'--cache-dir',str(env_paths['PIP_CACHE_DIR']),'--no-deps','--upgrade',*compatibility_specs],check=True)
    marker.write_text(json.dumps(expected,indent=2))
else:
    print('Compatibility stack already installed.')

subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(package_dir),'--cache-dir',str(env_paths['PIP_CACHE_DIR']),'google-auth>=2.35,<3','google-genai>=1.47,<2','pandas>=2.2,<3','numpy>=1.26,<3','PyYAML>=6,<7'],check=True)
site.addsitedir(str(package_dir)); sys.path.insert(0,str(package_dir)); sys.path.insert(0,str(PROJECT_DIR/'src')); importlib.invalidate_caches()

# Refuse cross-version module contamination instead of silently reusing an older ChemBreak stack.
expected_versions={'transformers':'4.40.2','tokenizers':'0.19.1','huggingface_hub':'0.23.5','accelerate':'0.30.1'}
for name, expected_version in expected_versions.items():
    mod=importlib.import_module(name)
    path=Path(mod.__file__).resolve()
    low=str(path).lower(); actual=str(getattr(mod,'__version__','?'))
    if 'chembreak' in low and 'chembreak14_storage' not in low:
        raise RuntimeError(f'{name} was already loaded from another ChemBreak storage: {path}. Restart the kernel and rerun CB14 from Cell 1.')
    if actual!=expected_version:
        raise RuntimeError(f'{name} version mismatch: expected {expected_version}, loaded {actual} from {path}. Restart the kernel and rerun CB14 from Cell 1.')
    print(f'{name}:',actual,'|',path)

import torch
print('torch:',torch.__version__,'| CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0),'| bf16:',torch.cuda.is_bf16_supported() if hasattr(torch.cuda,'is_bf16_supported') else 'unknown')


## C4 — Build the live runtime configuration

The repository contains a reproducible base YAML file. This cell changes only environment-specific paths and your Cloud project settings. No task split is changed here.

In [ ]:
import yaml
base=yaml.safe_load((PROJECT_DIR/'configs/config.cb14.yaml').read_text())
base['run'].update({'project_root':str(PROJECT_DIR),'task_bank_path':str(PROJECT_DIR/'data/final_task_bank.csv'),'mini_manifest_path':str(PROJECT_DIR/'data/CB14_mini24_manifest_v1.csv'),'mini_lock_path':str(PROJECT_DIR/'data/CB14_mini24_lock_v1.json'),'output_root':str(storage_root/'runs'),'dry_run':not LIVE,'experiment_revision':EXPERIMENT_REVISION,'task_limit':None,'live_progress':LIVE_PROGRESS})
base['targets'][0]['cache_dir']=str(model_cache); base['targets'][0]['offload_folder']=str(storage_root/'offload/ChemDFM')
policy_dir=storage_root/'policies'/EXPERIMENT_REVISION
base['policy']['training_artifact_path']=str(policy_dir/'training_policy.json'); base['policy']['frozen_artifact_path']=str(policy_dir/'frozen_policy.json')
runtime_path=storage_root/f'runtime_{EXPERIMENT_REVISION}.yaml'; runtime_path.write_text(yaml.safe_dump(base,sort_keys=False))
if LIVE:
    os.environ['GOOGLE_CLOUD_PROJECT']=PROJECT_ID
    os.environ['CHEMBREAK_ENABLE_LIVE']='YES'
print('Runtime config:',runtime_path)
print('LIVE:',LIVE,'| LIVE_PROGRESS:',LIVE_PROGRESS,'| Project:',os.environ.get('GOOGLE_CLOUD_PROJECT','mock'))

## C5 — Preflight

This checks the locked 24-task set, live project configuration, CUDA availability, ChemDFM tokenizer compatibility, and both short and long structured-output Vertex role probes before the experiment starts.

In [ ]:
from chembreak14.preflight import run_preflight
preflight=run_preflight(runtime_path,probe_tokenizer=LIVE,probe_roles=LIVE)
print(json.dumps(preflight,indent=2,sort_keys=True))
assert preflight['status']=='ok'

## C6 — Create the runner and load ChemDFM once

The same loaded ChemDFM instance is reused for Baseline, all three Learning epochs, and the Optimized phase. This avoids repeatedly reloading the 8B model.

In [ ]:
from chembreak14.runner import ChemBreak14Runner
runner=ChemBreak14Runner(runtime_path)
runner.load_target()
print('Runner ready.')
print('Planned episodes: 24 baseline + 72 learning + 24 optimized = 120')
print('Maximum target queries: 408 (usually lower because success stops an episode early)')

## C7 — Phase 1: Baseline

Each original benchmark prompt is sent to ChemDFM **once**. There is no adaptive strategy and no MDP update. The response is judged and stored. This gives **Baseline ASR** and also gives CB14 the initial response profile for each task.

With `LIVE_PROGRESS = True`, this cell prints each task as it starts, then immediately prints the judged one-turn result and the **running Baseline ASR** (`running_ASR`). Prompt and response bodies are not echoed to the notebook output.


In [ ]:
baseline_summary=runner.run_baseline()
print(json.dumps(baseline_summary,indent=2,sort_keys=True))

## C8 — Phase 2: Learning (3 epochs)

The same 24 tasks are run three times in **fresh conversations**. Each episode has at most four target turns. The MDP chooses exactly one abstract strategy per turn. Q-values and task memory carry forward from Epoch 1 → Epoch 2 → Epoch 3. Exploration decreases from 0.30 → 0.15 → 0.05.

With `LIVE_PROGRESS = True`, the cell prints a result line after **every target/judge turn** and a second line when each episode completes showing the running ASR for that learning epoch, overall progress, query count, elapsed time, and ETA.


In [ ]:
learning_summary=runner.run_learning()
print(json.dumps(learning_summary,indent=2,sort_keys=True))

## C9 — Freeze what the MDP learned

This creates the policy used for final optimized evaluation. After freezing, exploration is zero and the optimized phase is not allowed to update Q-values.


In [ ]:
frozen=runner.freeze_policy()
print(json.dumps(frozen,indent=2,sort_keys=True))

## C10 — Phase 3: Optimized evaluation

The same 24 benchmark **goals** are evaluated again in fresh conversations. CB14 does **not automatically send the original benchmark prompt first**. The frozen MDP uses the baseline response profile plus its general and task-specific learned memory to choose the first abstract strategy, then adapts after each target response for up to four turns.

With `LIVE_PROGRESS = True`, you see each turn result live and the **running Optimized ASR** after each completed task.


In [ ]:
optimized_summary=runner.run_optimized()
print(json.dumps(optimized_summary,indent=2,sort_keys=True))

## C11 — Results

The headline comparison is **Baseline ASR vs Optimized ASR**. The release also contains ASR for each learning epoch, mean turns, strategy usage, rewards, and the full stored turn log. The phase cells already show running metrics live; this Results cell prints the final summary and output-file locations.


In [ ]:
summary=runner.export_results()
print(json.dumps(summary,indent=2,sort_keys=True))
release_dir=storage_root/'runs'/EXPERIMENT_REVISION/'release'
print('Release directory:',release_dir)
print('Files:',[p.name for p in sorted(release_dir.iterdir())])

## C12 — Build a complete results ZIP and close the model

This ZIP contains the release CSV/JSON files plus the SQLite checkpoint, runtime configuration, fixed mini-set provenance files, and training/frozen policy artifacts. It deliberately excludes Hugging Face model weights and cache files.


In [ ]:
import zipfile
runner.close()
run_dir=storage_root/'runs'/EXPERIMENT_REVISION
release_dir=run_dir/'release'
zip_path=content_root/f'{EXPERIMENT_REVISION}_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in release_dir.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(run_dir))
    checkpoint=run_dir/'state.sqlite3'
    if checkpoint.exists(): z.write(checkpoint,Path('checkpoint')/checkpoint.name)
    if runtime_path.exists(): z.write(runtime_path,Path('provenance')/runtime_path.name)
    for p in [PROJECT_DIR/'data/CB14_mini24_manifest_v1.csv',PROJECT_DIR/'data/CB14_mini24_lock_v1.json']:
        z.write(p,Path('provenance')/p.name)
    for p in [policy_dir/'training_policy.json',policy_dir/'frozen_policy.json']:
        if p.exists(): z.write(p,Path('policies')/p.name)
print('Results ZIP:',zip_path)
print('Includes release tables, checkpoint, runtime config, mini-set lock/manifest, and policy artifacts.')
